# CNNによる画像認識（CIFAR10, PyTorch実装）


---
## 目的
PyTorch実装による畳み込みニューラルネットワーク（CNN）を用いてCIFAR10データセットに対する10クラスの物体認識を行う．
プログラムの構成はMNISTによる文字認識のプログラム（`mnist_cnn.ipynb`）と同様になっているため，基礎的な説明はそちらを参照して頂きたい．このページでは，MNISTによる文字認識のプログラムとの差分について書いていく．

また，GPUを用いたネットワークの計算を行う．

## 準備

### Google Colaboratoryの設定確認・変更
本チュートリアルではPyTorchを利用してニューラルネットワークの実装を確認，学習および評価を行います．
**GPUを用いて処理を行うために，上部のメニューバーの「ランタイム」→「ランタイムのタイプを変更」からハードウェアアクセラレータをGPUにしてください．**

## 使用するデータセット

### データセット
今回の物体認識では，CIFAR10データセットを用いる．CIFAR10データセットは，飛行機や犬などの10クラスの物体が表示されている画像から構成されたデータセットである．学習用画像は5万枚，評価用画像は1万枚であり，画像サイズは32×32のカラー画像（3チャンネル）となっている．

![CIFAR10](images/CIFAR10.png)

## モジュールのインポート
はじめに必要なモジュールをインポートする．

In [ ]:
from time import time   # 時間計測を行うモジュール

import torch            # 深層学習のモジュール（PyTorch）
import torch.nn as nn

import torchvision      # PyTorchで画像認識に使用するネットワークやデータセットを利用するためのモジュール
import torchvision.transforms as transforms

import torchsummary     # PyTorchで作成したネットワークモデルの詳細を表示するモジュール

### GPUの確認
GPUを使用した計算が可能かどうかを確認します．

`torch.cuda.is_available()`でGPUが利用可能かどうかを確認し，利用可能であれば`'cuda'`，利用できない場合は`'cpu'`を指定した`device`を作成します．

`device: cuda`と表示されれば，GPUを使用した計算をPyTorchで行うことが可能です．
`device: cpu`となっている場合は，上記の「Google Colaboratoryの設定確認・変更」に記載している手順にしたがって，設定を変更した後に，モジュールのインポートから始めてください．

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
学習データ（CIFAR10データセット）を読み込みます．

読み込んだ学習データのサイズを確認します．
学習データは5万枚，1つのデータサイズは3x32x32の画像のような形式となっています．
これは32x32ピクセルのカラー画像という意味になります．

In [ ]:
train_data = torchvision.datasets.CIFAR10(root="./", train=True, transform=transforms.ToTensor(), download=True)
test_data = torchvision.datasets.CIFAR10(root="./", train=False, transform=transforms.ToTensor(), download=True)

### 読み込んだデータの情報を表示して確認してみます
# 画像・ラベルデータのデータタイプ（型）
print(type(train_data.data), type(train_data.targets))
print(type(test_data.data), type(test_data.targets))
# 画像・ラベルの配列サイズ
print(train_data.data.shape, len(train_data.targets))
print(test_data.data.shape, len(test_data.targets))

### CIFAR10データセットの表示
CIFAR10データセットに含まれる画像を表示してみます．
ここでは，matplotlibを用いて複数の画像を表示させるプログラムを利用します．

In [ ]:
import matplotlib.pyplot as plt

cols = 10
rows = 2

plt.clf()
fig = plt.figure(figsize=(14, 4.8))
for r in range(rows):
    for c in range(cols):
        ax = fig.add_subplot(r+1, cols, c+1)
        ax.imshow(train_data[c+r*cols][0].permute(1, 2, 0))
        ax.set_axis_off()
plt.show()

## ネットワークモデルの定義

畳み込みニューラルネットワークを定義します．

ここでは，畳み込み層2層，全結合層2層から構成されるネットワークとします．

1層目の畳み込み層は入力チャンネル数が3，出力する特徴マップ数が8，畳み込むフィルタサイズが3x3です．
2層目の畳み込み層は入力チャネル数が8．出力する特徴マップ数が16，畳み込むフィルタサイズは同じく3x3です．
１つ目の全結合層は入力ユニット数は`8*8*16`とし，出力は64としています．
次の全結合層（出力層）は入力が64，出力が10です．
また，活性化関数として`self.act`にシグモイド関数を定義します．
さらに，プーリング処理を行うための`self.pool`を定義します．
ここでは，maxpoolingを使用します．
これらの各層の構成を`__init__`関数で定義します．

次に，`forward`関数では，定義した層を接続して処理するように記述します．
`forward`関数の引数`x`は入力データです．
それを`__init__`関数で定義した`conv1`に入力し，その出力を活性化関数である`self.act`に与えます．
そして，その出力を`self.pool`に与えて，プーリング処理結果を`h`として出力します．
2層目の畳み込み層でも同様の手順で処理を行います．

畳み込みを適用した後の特徴マップを全結合層へと入力して，識別結果を出力します．
まず．畳み込みによって得られた特徴マップの形状（チャンネルx縦x横）を1次元の配列へと変換します．
ここで，`view()`を用いることで，`h`の配列を操作します．引数として，変換したい配列のサイズを入力します．
まず一つ目の引数の`h.size()[0]`で，`h`の1次元目のサイズを取得し，変換後の配列の1次元目のサイズとして指定します．
二つ目の引数の`-1`で任意のサイズを指定します．
これにより，`h`を（バッチ数x任意の長さのデータ）の形状へ変換します．
変換した`h`を全結合層および活性化関数へと順次入力することで，最終的にクラススコアを返します．

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, stride=1, padding=1)
        self.l1 = nn.Linear(8 * 8 * 16, 64)
        self.l2 = nn.Linear(64, 10)
        self.act = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):                       # 入力画像サイズ: [batch, 3, 32, 32]
        h = self.pool(self.act(self.conv1(x)))  # 出力される特徴マップサイズ: [batch, 8, 16, 16]
        h = self.pool(self.act(self.conv2(h)))  # 出力される特徴マップサイズ: [batch, 16, 8, 8]
        # 特徴マップの配列サイズを [batch, channel, height, width] --> [batch, channel * height * width] に変更
        h = h.view(h.size()[0], -1)
        h = self.act(self.l1(h))
        h = self.l2(h)
        return h

## ネットワークの作成
上のプログラムで定義したネットワークを作成します．

`CNN`クラスを呼び出して，ネットワークモデルを定義します．
また，`.to(device)`を用いることで，ネットワークモデルを`device`で指定したデバイス（CPUまたはGPU）上に配置します．
これにより，`device`が`cuda`の場合はGPUを用いた演算が可能となります．

学習を行う際の最適化方法としてモーメンタムSGD(モーメンタム付き確率的勾配降下法）を利用します．
また，学習率を0.01，モーメンタムを0.9として引数に与えます．

最後に，定義したネットワークの詳細情報を`torchsummary.summary()`関数を用いて表示します．

In [ ]:
### ネットワークの作成
model = CNN()
model = model.to(device)  # ネットワークのパラメータをdeviceへ転送

### 最適化手法の設定 lr: 学習率, momentum: モーメンタム (慣性項のパラメータ)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

### 定義したモデルの情報を表示
torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFARデータセットと作成したネットワークを用いて，学習を行います．

1回の誤差を算出するデータ数（ミニバッチサイズ）を100，学習エポック数を10とします．

次にデータローダーを定義します．
データローダーでは，上で読み込んだデータセット（`train_data`）を用いて，for文で指定したミニバッチサイズでデータを読み込むオブジェクトを作成します．
この時，`shuffle=True`と設定することで，読み込むデータを毎回ランダムに指定します．

次に，誤差関数を設定します．
今回は，分類問題をあつかうため，クロスエントロピー誤差を計算するための`CrossEntropyLoss`を`criterion`として定義します．
ここでも，`.to(device)`を用いて誤差関数の計算をネットワークと同じデバイス上で行うように設定します．

学習を開始します．

各更新において，学習用データと教師データをそれぞれ`image`と`label`とします．
`image`と`label`は`.to(device)`を用いて，ネットワークと同じデバイスへ転送してから使用します．
学習モデルにimageを与えて各クラスの確率yを取得します．
各クラスの確率yと教師ラベルtとの誤差を`criterion`で算出します．
また，認識精度も算出します．
そして，誤差をbackward関数で逆伝播し，ネットワークの更新を行います．

In [ ]:
# ミニバッチサイズ・エポック数の設定
batch_size = 100
epoch_num = 10

# データローダーの設定
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

# 誤差関数の設定 (deviceに合わせて誤差関数の計算も同じデバイスで行うように設定)
criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

# ネットワークを学習モードへ変更
model.train()

# 学習の実行
train_start = time()
for epoch in range(1, epoch_num+1):    # epochのforループ
    # 1 epochの学習中の誤差・学習画像が正解した数をカウントする変数を初期化
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:  # 1 epoch内のforループ (iterationのループ)

        image = image.to(device)  # データを，モデルと同じdeviceへ転送
        label = label.to(device)

        y = model(image)  # データの入力と結果の出力

        # 誤差計算とbackpropagation, パラメータの更新
        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        # 学習経過を確認するための処理
        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    # 1 epoch終了時点での誤差の平均値，学習データに対する認識精度, 学習開始からの経過時間を表示
    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(epoch, sum_loss/len(train_loader), count.item()/len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
# データローダーの準備
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

# ネットワークを評価モードへ変更
model.eval()

# 評価の実行
count = 0
with torch.no_grad():  # 勾配を計算しない設定にする (loss.backwardをしても勾配情報が計算されない)
    for image, label in test_loader:

        image = image.to(device)  # データを，モデルと同じdeviceへ転送
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## 課題

### 1. ネットワークの構造を変更し，認識精度の変化を確認しましょう．

**ヒント：ネットワーク構造の変更としては，次のようなものが考えられます．**
* 中間層のユニット数，畳み込みのカーネルサイズ
* 層の数
* 活性化関数
  * `nn.Tanh()`や`nn.ReLU()`, `nn.LeakyReLU()`などが考えられます．
  * その他のPyTorchで使用できる活性化関数は[こちらページ](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity)にまとめられています．


### 2. 学習の設定を変更し，認識精度の変化を確認しましょう．

**ヒント：プログラムの中で変更で切る設定は次のようなものが存在します．**
* ミニバッチサイズ
* 学習回数（Epoch数）
* 学習率
* 最適化手法
  * `torch.optim.Adagrad()`や`torch.optim.Adam()`などが考えられます．
  * PyTorchで使用できる最適化手法は[こちらのページ](https://pytorch.org/docs/stable/optim.html#algorithms)にまとめられています．


### 3. [Optional] GPUを用いた場合とCPUを用いた場合の学習での計算時間の違いを確認しましょう．

GPUとCPUの切り替えは「GPUの確認」というセル（本ページ上部）にある

```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
```

の行を，例えば`device = torch.device('cpu')`のように書き換えることで，強制的にCPUを使用させることができます．